In [ ]:
from pipelines.readers.b3_indices_segmentos_setoriais import ReaderSQLB3IndicesSegmentosSetoriais

from yfinance import download
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

In [2]:
reader_b3_segmentos_setoriais = ReaderSQLB3IndicesSegmentosSetoriais()
df_b3_segmentos_setoriais = reader_b3_segmentos_setoriais.read(indice="IBEP")

print(df_b3_segmentos_setoriais.shape)
df_b3_segmentos_setoriais.tail(3)

(71, 7)


,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
68,IBEP,2026-07-07,VIVT3,TELEF BRASIL,ON EJ,707125712,1.169
69,IBEP,2026-07-07,WEGE3,WEG,ON NM,1485954732,3.293
70,IBEP,2026-07-07,YDUQ3,YDUQS PART,ON NM,261365845,0.108


In [3]:
reg_lin = {}

for codigo in df_b3_segmentos_setoriais['Código']:
    
    serie_precos = download(codigo + ".SA", period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)
    
    y = serie_precos["Adj Close"].to_numpy()
    x = np.arange(len(y))

    # regressão linear (grau 1)
    coef = np.polyfit(x, y, 1)      
    tendencia = np.polyval(coef, x)

    # Calcular o resíduo percentual
    residuo = ((serie_precos["Adj Close"] - tendencia) / tendencia) * 100
    
    residuo = residuo.iloc[-1]  # Pega o último valor do resíduo percentual
        
    reg_lin[codigo] = residuo

In [4]:
df = df_b3_segmentos_setoriais.copy()

df["residuo_reg_lin"] = df["Código"].map(reg_lin)

In [5]:
plot_df = (
    df[["Código", "residuo_reg_lin"]]
    .dropna()
    .sort_values("residuo_reg_lin")
)

fig = px.bar(
    plot_df,
    x="residuo_reg_lin",
    y="Código",
    orientation="h",
    color="residuo_reg_lin",
    color_continuous_scale="RdYlGn",
    labels={"residuo_reg_lin": "Variação (%)", "Código": "Empresa"},
    title="Variação percentual do preço em relação à tendência linear (últimos 10 anos)"
)

fig.add_vline(x=0, line_width=1, line_color="black")
fig.update_layout(height=1000, width=1200, yaxis={"categoryorder": "total ascending"})
fig.update_yaxes(tickmode="array", tickvals=plot_df["Código"], ticktext=plot_df["Código"])
fig.show()


In [6]:
print(f"Média do resíduo percentual em relação à tendência linear de todos os ativos: {df['residuo_reg_lin'].mean():.2f}")

Média do resíduo percentual em relação à tendência linear de todos os ativos: -7.84


In [7]:
print(f"Percentual de ativos com resíduo positivo em relação à tendência linear: {len(df[df['residuo_reg_lin'] > 0]) / len(df) * 100:.2f}%")

Percentual de ativos com resíduo positivo em relação à tendência linear: 57.75%


In [ ]:
# Calculo do resíduo percentual em relação à tendência linear para um ativo específico

serie_precos = download(ticker:="TAEE11.SA", period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)

y = serie_precos["Adj Close"].to_numpy()
x = np.arange(len(y))

coef = np.polyfit(x, y, 1)      # regressão linear (grau 1)
tendencia = np.polyval(coef, x)

residuo = ((serie_precos["Adj Close"] - tendencia) / tendencia) * 100

In [13]:
fig = go.Figure()

fig.add_scatter(
    x=serie_precos.index,
    y=serie_precos["Adj Close"],
    name="Preço"
)

fig.add_scatter(
    x=serie_precos.index,
    y=tendencia,
    name="Regressão Linear"
)

fig.show()

In [14]:
fig = go.Figure()

fig.add_scatter(
    x=serie_precos.index,
    y=residuo,
    name="Resíduo (%)"
)

fig.add_hline(
    y=0,
    line_color="red",
    line_width=1.5,
    line_dash="dash"
)

fig.show()

In [15]:

atual = residuo.iloc[-1]

# Estatísticas
media = residuo.mean()
mediana = residuo.median()
desvio = residuo.std()

p5 = residuo.quantile(0.05)
p25 = residuo.quantile(0.25)
p75 = residuo.quantile(0.75)
p95 = residuo.quantile(0.95)

zscore = (atual - media) / desvio

# Histograma
fig = px.histogram(
    x=residuo,
    nbins=50,
    title=f"{ticker} - Distribuição da distância para a tendência (%)",
    labels={
        "x": "Distância para a tendência (%)",
        "count": "Frequência"
    }
)

# Média
fig.add_vline(
    x=media,
    line_color="blue",
    line_width=2,
)

# Mediana
fig.add_vline(
    x=mediana,
    line_color="green",
    line_dash="dash",
    line_width=2,
)

# ±1 desvio padrão
fig.add_vline(
    x=media + desvio,
    line_color="gray",
    line_dash="dot",
    annotation_text="+1σ",
)

fig.add_vline(
    x=media - desvio,
    line_color="gray",
    line_dash="dot",
    annotation_text="-1σ",
)

# Percentis
fig.add_vline(
    x=p5,
    line_color="orange",
    line_dash="dash",
    annotation_text="P5"
)

fig.add_vline(
    x=p95,
    line_color="orange",
    line_dash="dash",
    annotation_text="P95"
)

# Linha do zero
fig.add_vline(
    x=0,
    line_color="black",
    line_width=1,
    line_dash="dash"
)

# Valor atual
fig.add_vline(
    x=atual,
    line_color="red",
    line_width=3,
    annotation_text=f"Atual ({atual:.2f}%)",
    annotation_position="top"
)

# Caixa de estatísticas
fig.add_annotation(
    xref="paper",
    yref="paper",
    x=0.99,
    y=0.99,
    xanchor="right",
    yanchor="top",
    showarrow=False,
    align="left",
    bgcolor="white",
    bordercolor="black",
    text=(
        f"<b>Média:</b> {media:.2f}%<br>"
        f"<b>Mediana:</b> {mediana:.2f}%<br>"
        f"<b>Desvio:</b> {desvio:.2f}%<br>"
        f"<b>Z-Score:</b> {zscore:.2f}<br>"
        f"<b>P5:</b> {p5:.2f}%<br>"
        f"<b>P25:</b> {p25:.2f}%<br>"
        f"<b>P75:</b> {p75:.2f}%<br>"
        f"<b>P95:</b> {p95:.2f}%"
    )
)

fig.update_layout(
    width=1100,
    height=600,
    bargap=0.05
)

fig.show()